In [ ]:
import pandas as pd

# 1. Import des données

In [ ]:
df_irve = pd.read_csv(
 'https://www.data.gouv.fr/api/1/datasets/r/eb76d20a-8501-400e-b336-d85724de5435'
)

print(f"Shape : {df_irve.shape}")
df_irve.head()

Ce premier jeu de données est la base nationale des IRVE (Infrastructures de Recharge pour Véhicules Électriques).
Cette table comporte 210129 lignes pour 52 variables.

In [ ]:
df_ve = pd.read_csv(
 'https://www.data.gouv.fr/api/1/datasets/r/90e0d717-deda-4bdc-9987-f82faac5bc93',
 encoding='latin-1'
)

print(f"Shape : {df_ve.shape}")
df_ve.head()

Ce second jeu de données contient des informations sur les voitures particulières immatriculées par commune et par type de recharge.
Cette table comporte 703545 lignes pour 8 variables.

In [ ]:
df_revenus = pd.read_csv(
 'https://www.data.gouv.fr/api/1/datasets/r/516130bc-4dcb-47f5-8347-ae96553c43ab',sep=';'
)

print(f"Shape : {df_revenus.shape}")
df_revenus.head()

Ce dernier jeu de données contient des informations sur les voitures particulières immatriculées par commune et par type de recharge.
Cette table comporte 34926 lignes pour 57 variables.

# 2. Choix de la variable de jointure

In [ ]:
list(df_irve.columns)

In [ ]:
list(df_ve.columns)

In [ ]:
list(df_revenus.columns)

Nous remarquons que ces 3 jeu de données contiennent une variable indiquant le code géographique mais sous 3 noms différents :
- `code_insee_commune` pour df_irve
- `CODGEO` pour df_ve
- `Code géographique` pour df_revenus

# 3. Préparation de la variable de jointure

In [ ]:
from src.preparation_data import diagnostic_cle_jointure

In [ ]:
diagnostic_cle_jointure(df_irve,"code_insee_commune","df_irve")
diagnostic_cle_jointure(df_ve,"CODGEO","df_ve")
diagnostic_cle_jointure(df_revenus,"Code géographique","df_revenus")

Ce diagnostic met en évidence plusieurs points d’attention :

- Présence de valeurs manquantes dans `code_insee_commune` (df_irve)  
- Présence de longueurs de codes incohérentes :
  - `code_insee_commune` (df_irve) contient des codes de longueur 3, 6 et 7
  - `CODGEO` (df_ve) contient des codes de longueur 4
- Les variables de jointure ne sont pas uniques :
  - `code_insee_commune` (df_irve)
  - `CODGEO` (df_ve)

Les clés de jointure présentent des problèmes de qualité et de format.  
Il est nécessaire de les nettoyer et les uniformiser (type, format, gestion des valeurs manquantes, unicité) avant d’effectuer la jointure.

In [ ]:
taile_louche = df_irve[
    df_irve["code_insee_commune"].notna() &
    (df_irve["code_insee_commune"].astype(str).str.len() == 7)
]
taile_louche["code_insee_commune"]

longueur 3 c'est les 0.0
longueur 6 c'est les codes à 4 chiffres suivis de .0
longueur 7 c'est les codes à 5 chiffres suivis de .0

In [ ]:
taile_louche = df_ve[
    df_ve["CODGEO"].notna() &
    (df_ve["CODGEO"].astype(str).str.len() == 4)
]
taile_louche["CODGEO"]

longueur 4 c'est les codes à 4 chiffres

L'objectif maintenant est d'uniformiser ces codes géographiques pour qu'ils soient tous à 5 chiffres.